# **🎬 IMDB Movie Review Sentiment Analysis – **Training + Inference****

This notebook implements both **training** and **inference** workflows for a deep learning–based movie review sentiment classifier using the **IMDB Dataset**. The process is divided into two independent modules:

1. **Training Module** — responsible for preparing data, building the model, training it, and saving the necessary artifacts.
2. **Inference Module** — loads the saved model + tokenizer and performs instant predictions on new movie reviews.

Both modules are explained clearly below.

---

## 🧠 **Part 1 — Training Pipeline (LSTM Deep Learning Model)**

This section performs a complete **text preprocessing + tokenization + sequence padding + LSTM model training** workflow using the IMDB Movie Reviews dataset.
The goal is to train a Deep Learning classifier that can distinguish **Positive** and **Negative** movie reviews.

---

### 🔎 **What the Training Code Does**

#### **1️⃣ Loads and Inspects the Dataset**

* Reads `IMDB_Dataset.csv`
* Prints the first and last few rows for inspection
* Drops any missing or corrupt rows

---

#### **2️⃣ Cleans All Movie Reviews**

A custom `clean_text()` function:

* Converts text to lowercase
* Removes HTML tags
* Removes special characters, punctuation, and non-alphabetic characters
* Collapses extra whitespace
* Produces uniform, clean text ready for NLP processing

---

#### **3️⃣ Prepares Features (X) and Labels (y)**

* Converts cleaned reviews into the `X` variable
* Maps sentiments:

  * `positive → 1`
  * `negative → 0`
* Splits into **80% training** and **20% testing** sets

---

#### **4️⃣ Tokenization + Sequence Padding**

* Builds a tokenizer limited to the **10,000 most frequent words**
* Fits tokenizer on the training text
* Converts all text into integer sequences
* Pads sequences to a fixed length of **200 tokens**
* Ensures consistent shape for LSTM input
* Saves the tokenizer as `imdb_tokenizer.json` for later reuse

---

#### **5️⃣ Builds the LSTM-Based Deep Learning Model**

The model architecture:

* **Embedding Layer (10k × 128)**
* **LSTM Layer (128 units)** with dropout + recurrent dropout
* **Dense Layer (1 unit, sigmoid)** for binary sentiment classification

The model is compiled with:

* **Binary Crossentropy Loss**
* **Adam Optimizer**
* **Accuracy Metric**

---

#### **6️⃣ Trains the Sentiment Model**

* Trains for **5 epochs** with batch size **64**
* Uses **10% of training data** as validation
* Learns sentiment patterns from 40,000 reviews
* Achieves ~88% accuracy on validation set

---

#### **7️⃣ Saves the Trained Model**

After training:

* Model saved as **`imdb_lstm_sentiment.h5`**
* Tokenizer already saved to `imdb_tokenizer.json`
* These two files are all you need for inference

This ensures you **never need to retrain again** unless you want improved accuracy.

---

#### **8️⃣ Evaluates on Test Set**

* Converts test text into padded sequences
* Predicts sentiment labels
* Displays:

  * **Accuracy**
  * **Precision, Recall, F1-score**
  * Full Classification Report

This verifies how well the model generalizes to unseen reviews.

---

### 🎯 **Training Outcome**

By the end of the training section, you will have:

* A **fully trained and saved LSTM model** (`imdb_lstm_sentiment.h5`)
* A **saved tokenizer** (`imdb_tokenizer.json`)
* A verified accuracy of ~88% on unseen data
* The entire processing pipeline preserved for future use
* Zero need for retraining when running inference

---

## ⚡ **Part 2 — Inference Pipeline (Using Saved Model + Tokenizer)**

This section uses the previously trained model to make **instant predictions** on new user-entered reviews.
No training, no dataset split, no preprocessing of the full dataset — **just pure prediction**.

---

### 🔎 **What the Inference Code Does**

#### **1️⃣ Loads the Saved Tokenizer**

* Loads `imdb_tokenizer.json`
* Restores the exact word-index mapping used during training
* Ensures consistency between train-time and inference-time tokenization

---

#### **2️⃣ Loads the Pre-Trained LSTM Model**

* Loads `imdb_lstm_sentiment.h5`
* Restores all trained weights:

  * Embedding matrix
  * LSTM parameters
  * Output classifier

Prediction becomes instant with **no additional training**.

---

#### **3️⃣ Cleans the User’s Input Review**

Applies the same cleaning steps used during training:

* Lowercase
* HTML removal
* Non-alphabet cleanup
* Whitespace normalization

Ensures input is compatible with training-style text.

---

#### **4️⃣ Converts Review Text Into Padded Sequences**

* Tokenizes the cleaned text
* Converts words → integer indices
* Pads to **200 tokens**
* Creates a proper model input tensor

---

#### **5️⃣ Predicts Sentiment (Positive / Negative / Mixed)**

The LSTM outputs a probability:

* **> 0.60 → Positive**
* **< 0.40 → Negative**
* **0.40–0.60 → Uncertain / Mixed**

This avoids misclassifying borderline cases.

---

#### **6️⃣ Smart Input Validation**

The script prevents meaningless predictions:

* Rejects empty inputs
* Rejects one-word inputs
* Rejects reviews containing only unknown words
* Asks user to provide at least **2–3 meaningful words**

This ensures stable and reliable sentiment analysis.

---

### 🎯 **Inference Outcome**

By the end of the inference section, you have:

* A ready-to-use **Sentiment Prediction Engine**
* Zero retraining required
* Clean, interactive CLI interface
* Accurate predictions on any movie review you enter
* Lightning-fast performance using saved weights

---

## 🚀 **Complete Project Summary**

This notebook demonstrates a full end-to-end sentiment analysis system using Deep Learning:

#### ✔ Data Cleaning

#### ✔ Tokenization & Padding

#### ✔ LSTM Model Building

#### ✔ Training & Validation

#### ✔ Saving Model + Tokenizer

#### ✔ Reusable Inference Script

#### ✔ Real-Time Review Classification

You now have a **production-ready NLP sentiment classifier** that can be embedded into:

* Web apps (Streamlit / Flask / FastAPI)
* Desktop apps
* Mobile APIs
* Chatbots
* Data pipelines

Your IMDB Sentiment Project is now **complete, reusable, and deployable**.


In [1]:
import os
import re
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer, tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# -------------------------------------------------
# 0. Reproducibility (optional)
# -------------------------------------------------
np.random.seed(42)
tf.random.set_seed(42)

# Paths for saving model and tokenizer
MODEL_PATH = "imdb_lstm_sentiment.h5"
TOKENIZER_PATH = "imdb_tokenizer.json"

# -------------------------------------------------
# 1. Load and inspect data
# -------------------------------------------------
print("Loading dataset ...")

data = pd.read_csv("IMDB_Dataset.csv")

# Drop rows with missing review or sentiment
data = data.dropna(subset=["review", "sentiment"])

print("Dataset head:")
print(data.head())
print("\nDataset tail:")
print(data.tail())

# -------------------------------------------------
# 2. Text cleaning function
# -------------------------------------------------

def clean_text(text: str) -> str:
    """Lowercase, strip HTML, keep only letters, collapse spaces."""
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)
    # Keep only letters
    text = re.sub(r"[^a-zA-Z]", " ", text)
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)
    return text.strip()


print("\nCleaning reviews ...")
data["clean_review"] = data["review"].apply(clean_text)

# -------------------------------------------------
# 3. Prepare X and y
# -------------------------------------------------
X = data["clean_review"]
y = data["sentiment"].map({"positive": 1, "negative": 0}).values

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("\nTrain size:", len(X_train_text))
print("Test size:", len(X_test_text))

# -------------------------------------------------
# 4. Tokenization & Padding (with save/load of tokenizer)
# -------------------------------------------------
max_words = 10000
max_len = 200

if os.path.exists(TOKENIZER_PATH):
    print("\nLoading existing tokenizer from file ...")
    with open(TOKENIZER_PATH, "r", encoding="utf-8") as f:
        tokenizer_data = json.load(f)
    tokenizer = tokenizer_from_json(tokenizer_data)
else:
    print("\nFitting new tokenizer on training data ...")
    tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
    tokenizer.fit_on_texts(X_train_text)

    # Save tokenizer to JSON
    tokenizer_json = tokenizer.to_json()
    with open(TOKENIZER_PATH, "w", encoding="utf-8") as f:
        f.write(tokenizer_json)
    print(f"Tokenizer saved to {TOKENIZER_PATH}")

# Convert text to sequences
print("\nConverting text to padded sequences ...")
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len,
    padding="post",
    truncating="post",
)
X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len,
    padding="post",
    truncating="post",
)

# Effective vocabulary size (for Embedding layer)
vocab_size = min(max_words, len(tokenizer.word_index) + 1)
print("Vocabulary size (used in Embedding):", vocab_size)

# -------------------------------------------------
# 5. Build / Load the LSTM model
# -------------------------------------------------

if os.path.exists(MODEL_PATH):
    print(f"\nLoading existing model from {MODEL_PATH} ...")
    model = load_model(MODEL_PATH)
else:
    print("\nBuilding a new LSTM model ...")
    model = Sequential()
    model.add(Embedding(input_dim=vocab_size, output_dim=128))
    model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        loss="binary_crossentropy",
        optimizer="adam",
        metrics=["accuracy"],
    )

    # Optional: build for a nicer summary
    model.build(input_shape=(None, max_len))

    print("\nModel summary:")
    print(model.summary())

    # -------------------------------------------------
    # 6. Train the model (only when not already saved)
    # -------------------------------------------------
    print("\nTraining model ...")
    history = model.fit(
        X_train_pad,
        y_train,
        epochs=5,
        batch_size=64,
        validation_split=0.1,
        verbose=1,
    )

    # Save the trained model
    model.save(MODEL_PATH)
    print(f"\nModel saved to {MODEL_PATH}")

# -------------------------------------------------
# 7. Evaluate on test set
# -------------------------------------------------
print("\nEvaluating on test set ...")

y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int).ravel()

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))

# -------------------------------------------------
# 8. Interactive prediction loop with sanity checks
# -------------------------------------------------
print("\nSentiment analyzer ready.\n")

while True:
    user_input = input("Enter a movie review (or type 'exit' to quit):\n")
    user_input = user_input.strip()

    # Exit condition
    if user_input.lower() == "exit":
        print("Exiting sentiment analyzer. Goodbye!")
        break

    # 1) Basic empty check
    if user_input == "":
        print("Please type a proper review sentence, not an empty input.\n")
        continue

    # 2) Clean and check length
    cleaned_input = clean_text(user_input)
    if cleaned_input == "":
        print("I couldn't understand that. Try using normal English words.\n")
        continue

    tokens = cleaned_input.split()
    if len(tokens) < 4:
        print("Please enter at least 4–5 words so I can analyze the sentiment.\n")
        continue

    # 3) Tokenize and check if anything is recognized
    input_seq = tokenizer.texts_to_sequences([cleaned_input])

    if len(input_seq[0]) == 0:
        print("I couldn't map your input to known words. Try a more detailed review.\n")
        continue

    input_pad = pad_sequences(
        input_seq,
        maxlen=max_len,
        padding="post",
        truncating="post",
    )

    # 4) Predict
    prediction = model.predict(input_pad)[0][0]

    # 5) Use a small "uncertain" band
    if prediction > 0.6:
        sentiment = "Positive"
    elif prediction < 0.4:
        sentiment = "Negative"
    else:
        sentiment = "Uncertain / Mixed"

    print(f"\nPredicted Sentiment: {sentiment}")
    print(f"Confidence: {prediction:.2f}\n")


Loading dataset ...
Dataset head:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Dataset tail:
                                                  review sentiment
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

Cleaning reviews ...

Train size: 40000
Test size: 10000

Fitting new tokenizer on training data ...
Tokenizer saved to imdb_tokenizer.json

Converting text to padded seq

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 200, 128)            │       1,280,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 128)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,411,713 (5.39 MB)

 Trainable params: 1,411,713 (5.39 MB)

...

Please enter at least 4–5 words so I can analyze the sentiment.



Enter a movie review (or type 'exit' to quit):
 It was a very great movie


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step

Predicted Sentiment: Positive
Confidence: 0.95



Enter a movie review (or type 'exit' to quit):
 exit


Exiting sentiment analyzer. Goodbye!


In [6]:
import re
import tensorflow as tf

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.models import load_model

# -----------------------------
# 1. Paths
# -----------------------------
TOKENIZER_PATH = "imdb_tokenizer.json"
MODEL_PATH = "imdb_lstm_sentiment.h5"
max_len = 200   # must match training time

# -----------------------------
# 2. Load tokenizer
# -----------------------------
with open(TOKENIZER_PATH, "r", encoding="utf-8") as f:
    tokenizer_json = f.read()

tokenizer = tokenizer_from_json(tokenizer_json)

# -----------------------------
# 3. Load trained model
# -----------------------------
model = load_model(MODEL_PATH)
print("Model and tokenizer loaded successfully!")


# -----------------------------
# 4. Cleaning function
# -----------------------------
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# -----------------------------
# 5. Prediction function
# -----------------------------
def predict_sentiment(review: str):
    cleaned = clean_text(review)

    if cleaned == "":
        return "Invalid input (no usable text)", 0.0

    seq = tokenizer.texts_to_sequences([cleaned])
    if len(seq[0]) == 0:
        return "Input not recognized (all OOV)", 0.0

    pad = pad_sequences(
        seq,
        maxlen=max_len,
        padding="post",
        truncating="post"
    )

    pred = model.predict(pad)[0][0]

    if pred > 0.6:
        sentiment = "Positive"
    elif pred < 0.4:
        sentiment = "Negative"
    else:
        sentiment = "Uncertain / Mixed"

    return sentiment, float(pred)


# -----------------------------
# 6. Interactive loop
# -----------------------------
while True:
    user_input = input("\nEnter review (or 'exit'): ").strip()

    if user_input.lower() == "exit":
        print("Goodbye!")
        break

    if len(user_input.split()) < 2:
        print("Please enter at least 2–3 meaningful words.")
        continue

    sentiment, confidence = predict_sentiment(user_input)

    print(f"\nSentiment: {sentiment}")
    if confidence > 0:
        print(f"Confidence: {confidence:.2f}")


Model and tokenizer loaded successfully!



Enter review (or 'exit'):  The cinematography was beautiful and the soundtrack was great, but the story felt slow and the ending was disappointing.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step

Sentiment: Uncertain / Mixed
Confidence: 0.49



Enter review (or 'exit'):  exit


Goodbye!
